# Run NIMBO in Google Colab

Choose **Runtime → Change runtime type → CPU**, connect to a free runtime, then run the cells in order. A Google sign-in may be needed to connect. Setup downloads public inputs and dependencies to temporary storage. Download the reviewed ZIP before the session ends.

NIMBO runs the same pinned DistilBERT classifier through PyTorch FP32 and ONNX Runtime FP32. It exports the graph, benchmarks both runtimes, checks quality on a separate labeled sample, and verifies a runnable ZIP if the candidate passes. The notebook shows measurements from your run.

## 1. Prepare the CPU runtime

This cell uses an isolated Python environment, the verified NIMBO source commit, and the public model and SST-2 revisions below. The initial download can take a few minutes.

- [DistilBERT model revision](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/714eb0fa89d2f80546fda750413ed43d93601a13)
- [SST-2 dataset revision](https://huggingface.co/datasets/stanfordnlp/sst2/tree/8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb)

In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import platform
import shutil
import signal
import subprocess
import sys
import tempfile
import zipfile

SOURCE_COMMIT = "5dc6e74ded479989d406a922e80793b96cef9cd7"
SOURCE_URL = "https://github.com/LeenAlaslani/NIMBO.git"

if sys.platform != "linux" or platform.machine() != "x86_64":
    raise RuntimeError("Use a Linux x86-64 CPU runtime, such as Colab.")
if sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Select a Colab runtime with Python 3.12 or 3.13 in Change runtime type.")
if shutil.disk_usage(tempfile.gettempdir()).free < 4 * 1024**3:
    raise RuntimeError("The session needs at least 4 GiB of free temporary storage.")

# This directory lives until the notebook kernel exits or setup is rerun.
SESSION = tempfile.TemporaryDirectory(prefix="nimbo-colab-")
ROOT = Path(SESSION.name)
SOURCE, INPUTS = ROOT / "source", ROOT / "inputs"
VENV = ROOT / "venv"
PYTHON = VENV / "bin" / "python"

public_env = os.environ.copy()
for key in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGING_FACE_HUB_TOKEN", "GH_TOKEN", "GITHUB_TOKEN"):
    public_env.pop(key, None)
public_env.update({
    "HF_HOME": str(ROOT / "hf"),
    "HF_HUB_CACHE": str(ROOT / "hf" / "hub"),
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "HF_HUB_DISABLE_XET": "1",
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_GLOBAL": os.devnull,
})
for key in ("HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE", "TRANSFORMERS_OFFLINE"):
    public_env.pop(key, None)

def run(argv, *, cwd=None, env=public_env):
    # IPython captures direct subprocess stdout inconsistently across runtimes.
    # Forward the lines so notebook users can watch each setup and job stage.
    with subprocess.Popen([str(arg) for arg in argv], cwd=cwd, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as proc:
        try:
            for line in proc.stdout:
                print(line.rstrip("\n"), flush=True)
        except KeyboardInterrupt:
            proc.send_signal(signal.SIGINT)
            raise
        if proc.wait() != 0:
            raise subprocess.CalledProcessError(proc.returncode, argv)

SOURCE.mkdir()
run(["git", "init", "-q"], cwd=SOURCE)
run(["git", "fetch", "-q", "--depth=1", SOURCE_URL, SOURCE_COMMIT], cwd=SOURCE)
run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], cwd=SOURCE)
checked_out = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True).strip()
if checked_out != SOURCE_COMMIT:
    raise RuntimeError("The fetched source commit did not match.")

# pip runs from the notebook kernel and installs into the isolated environment.
# This also works when the base runtime lacks ensurepip for virtual environments.
run([sys.executable, "-m", "venv", "--without-pip", VENV])
pip = [sys.executable, "-m", "pip", "--python", PYTHON]
run([*pip, "install", "--disable-pip-version-check",
     "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.9.1+cpu"])
run([*pip, "install", "--disable-pip-version-check", "-r",
     SOURCE / "deploy" / "reviewer-requirements.txt", "pyarrow==23.0.1"])
run([*pip, "check"])

run([PYTHON, "-m", "tools.prepare_nimbo_reviewer", INPUTS], cwd=SOURCE)
prepared = json.loads((INPUTS / "model.json").read_text())
heldout = json.loads((INPUTS / "heldout.source.json").read_text())
print(f"Model {prepared['model_id']} at {prepared['revision']}")
print(f"Data {heldout['source']} at {heldout['revision']}")
print(f"NIMBO source {checked_out}")

## 2. Run a fresh optimization

Each path uses one CPU thread. The pilot uses 24 labeled training examples, followed by three paired rounds on 200 separate validation examples when the pilot passes. SST-2 is the model's source task. Benchmark timing varies with the runtime.

In [ ]:
from uuid import uuid4

result = None
bundle = None
OUT = ROOT / f"run-{uuid4().hex}"
OUT.mkdir()

# The child invokes the same supervisor and deterministic worker as the reviewer image.
# It copies the review ZIP out before the supervisor cleans up its per-job workspace.
runner = """
import json
from pathlib import Path
import shutil
import sys
import time

from app import nimbo_reviewer as reviewer

inputs, output = map(Path, sys.argv[1:])
reviewer.WORKER_RSS = 2_500 * reviewer.MIB
service = reviewer.Reviewer(output / "state", inputs)
service.start()
try:
    accepted = service.submit("notebook")
    job_id = accepted["job_id"]
    print(f"Fresh job {job_id}", flush=True)
    seen = 0
    while True:
        with service.lock:
            job = service.jobs[job_id]
            events = job.events[seen:]
            seen += len(events)
            status = job.status
            package = job.package
        for event in events:
            print(f"[{event['stage']}] {event['message']}", flush=True)
        if status["state"] in reviewer.engine.FINAL:
            break
        time.sleep(0.5)
    if status["eligible_for_review_download"]:
        shutil.copyfile(package, output / "nimbo-verified-review.zip")
    (output / "receipt.json").write_text(json.dumps(status, indent=2))
finally:
    service.close()
"""
offline_env = {**public_env, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
               "HF_DATASETS_OFFLINE": "1", "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}
run([PYTHON, "-c", runner, INPUTS, OUT], cwd=SOURCE, env=offline_env)
result = json.loads((OUT / "receipt.json").read_text())

def format_metric(metrics, key, multiplier=1):
    value = metrics.get(key)
    return "Unavailable" if value is None else f"{value * multiplier:,.2f}"

report = [f"**Job `{result['job_id']}`**", f"**Decision** `{result['decision']}`", result["reason"]]
paired = result["comparison"]
if paired:
    split = "Held-out" if paired["scope"] == "heldout" else "Pilot"
    rows = [
        ("Mean warm forward latency (ms)", "mean_warm_forward_ms", 1),
        ("p95 warm forward latency (ms)", "p95_warm_forward_ms", 1),
        ("Total process CPU time (ms)", "process_cpu_forward_total_ms", 1),
        ("Labeled accuracy (%)", "accuracy", 100),
    ]
    report.extend([
        f"\n{split} comparison on {paired['examples']} examples over {paired['rounds']} paired rounds.\n",
        "| Metric | PyTorch FP32 | ONNX Runtime FP32 |",
        "| --- | ---: | ---: |",
    ])
    for label, key, multiplier in rows:
        baseline = format_metric(paired["baseline"], key, multiplier)
        candidate = format_metric(paired["candidate"], key, multiplier)
        report.append(f"| {label} | {baseline} | {candidate} |")
    gates = paired["gates"]
    def verdict(key):
        value = gates.get(key)
        return "passed" if value is True else "failed" if value is False else "unavailable"
    report.append(
        f"\nQuality {verdict('quality_pass')}. Performance {verdict('performance_pass')}. "
        f"Prediction agreement {format_metric(gates, 'prediction_agreement', 100)}%."
    )
try:
    from IPython.display import Markdown, display
except ImportError:
    print("\n\n".join(report))
else:
    display(Markdown("\n\n".join(report[:3]) + "\n" + "\n".join(report[3:])))
if result["eligible_for_review_download"]:
    bundle = OUT / "nimbo-verified-review.zip"

## 3. Independently verify and download the review package

When the candidate passes, this cell checks the archive checksum and independently reloads the extracted ONNX package on the same labeled validation sample. The verified ZIP contains the model, tokenizer, manifest, evidence and a local runner. It is approximately 268 MB.

In [ ]:
if bundle is None:
    print("This run did not produce a verified review package.")
else:
    with bundle.open("rb") as stream:
        checksum = hashlib.file_digest(stream, "sha256").hexdigest()
    if checksum != result["package"]["sha256"]:
        raise RuntimeError("The exported archive checksum did not match the job receipt.")

    with tempfile.TemporaryDirectory(dir=OUT, prefix="verify-") as directory:
        with zipfile.ZipFile(bundle) as archive:
            names = [PurePosixPath(name) for name in archive.namelist()]
            if (not names or any(name.is_absolute() or len(name.parts) != 2 or ".." in name.parts
                                 for name in names) or len({name.parts[0] for name in names}) != 1):
                raise RuntimeError("Unexpected paths in the generated archive.")
            archive.extractall(directory)
        extracted = Path(directory) / names[0].parts[0]
        run([PYTHON, "-m", "tools.verify_nimbo_package_v1", "--package", extracted,
             "--dataset", INPUTS / "heldout.csv"], cwd=SOURCE, env=offline_env)

    print(f"Verified SHA-256 {checksum}")
    try:
        from google.colab import files
    except ImportError:
        print(f"Verified package saved to {bundle}")
    else:
        files.download(str(bundle))